In [ ]:
import pandas as pd

from os import path
import re
import itertools

from pymodulon.io import load_json_model
from pymodulon.plotting import *

import numpy as np
import plotly.express as px
import seaborn as sns

from scipy.stats import pearsonr

from tqdm.notebook import tqdm

from collections import defaultdict

from matplotlib.lines import Line2D
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['svg.fonttype'] = 'none'
matplotlib.rcParams['font.sans-serif'] = 'Arial'
matplotlib.rcParams['font.family'] = 'sans-serif'
sns.set_style('ticks')
matplotlib.rcParams['text.color'] = '#000000'
matplotlib.rcParams['axes.labelcolor'] = '#000000'
matplotlib.rcParams['xtick.color'] = '#000000'
matplotlib.rcParams['ytick.color'] = '#000000'

In [ ]:
from pymodulon.compare import *

In [ ]:
ica_data = load_json_model('../../modulome_sc.json')

In [ ]:
ica_data_cenpk = load_json_model('../../../modulome_sc_cenpk_2025/modulome_sc_cenpk.json')

In [ ]:
cenpk_out = "../../../modulome_sc_cenpk_2025/data/bbh_s288c/prot_cenpk.fasta"
s288c_out = "../../../modulome_sc_cenpk_2025/data/bbh_s288c/prot_s288c.fasta"
bbh = get_bbh(cenpk_out,s288c_out, outdir = "../../../modulome_sc_cenpk_2025/data/bbh_s288c")

In [ ]:
links,dots = compare_ica(ica_data_cenpk.M,ica_data.M,
                         ortho_file = bbh, method = 'pearson', cutoff = .3)

dots.render("images/cenpk_s288c_ica_comparison", format="svg", cleanup=True)
dots

In [ ]:
from graphviz import Graph
from IPython.display import display, SVG

In [ ]:
names1 = [x[0] for x in links]
names2 = [x[1] for x in links]

In [ ]:
# Dataset1 names
name_dict1 = {}
for name in names1:
    val = str(name)
    if len(val) > 10:
        name_dict1[name] = val[: len(val) // 2] + "\n" + val[len(val) // 2 :]
    else:
        name_dict1[name] = val

# Dataset2 names
name_dict2 = {}
for name in names2:
    val = str(name)
    if len(val) > 10:
        name_dict2[name] = val[: len(val) // 2] + "\n" + val[len(val) // 2 :]
    else:
        name_dict2[name] = val


In [ ]:
len(links)

# Plots for Cen.pk Data

In [ ]:
im = 'IFH1'
s288c_genes = ica_data.view_imodulon(im).index
cenpk_genes = ica_data_cenpk.view_imodulon(im).index
overlap = [x for x in ica_data_cenpk.view_imodulon(im).index if x in bbh.gene.values and bbh.set_index('gene').loc[x,'subject'] in s288c_genes]
all_genes = ica_data_cenpk.X.index

def categorize_genes(x):
    if x in overlap:
        return 'Has Ortholog in S288C'
    elif x in cenpk_genes and x in bbh.gene.values:
        return 'Ortholog not in S288C Component'
    elif x in cenpk_genes:
        return "No Known Ortholog"
    return 'Not in iM'

groups = {x:categorize_genes(x) for x in all_genes}
kwargs = {'groups':groups, 'colors':{'Has Ortholog in S288C': 'orange', "No Known Ortholog":'tab:blue', 'Not in iM':'Gray', 'Ortholog not in S288C Component':'tab:purple'}}

In [ ]:
plot_gene_weights_chrom(ica_data_cenpk, im, **kwargs)
plt.xticks(rotation = 45)
plt.title('Cen.PK IFH1 iModulon');
plt.savefig('images/fig5_ifh1_cenpk.svg', dpi = 600, format = 'svg', bbox_inches='tight')

In [ ]:
from matplotlib_venn import venn2
# Define the sizes of the regions
size_only_A = len(cenpk_genes) - len(overlap)  # Elements only in Set A
size_only_B = len(s288c_genes) - len(overlap)   # Elements only in Set B
size_A_and_B = len(overlap)  # Elements in both Set A and Set B

# Create the Venn diagram
venn2(subsets=(size_only_A, size_only_B, size_A_and_B), set_labels=(f'CEN.PK {im} iM Genes', f'S288C {im} iM Genes'))

plt.title('CEN.PK IFH1 Overlap With S288C');


plt.savefig('images/fig5_ifh1_cenpk_vs_s288c_venn.svg', dpi = 600, format = 'svg', bbox_inches='tight')

# Display the diagram
plt.show()

In [ ]:
im = 'GCN4 Associated'
s288c_genes = ica_data.view_imodulon(im).index
cenpk_genes = ica_data_cenpk.view_imodulon(im).index
overlap = [x for x in ica_data_cenpk.view_imodulon(im).index if x in bbh.gene.values and bbh.set_index('gene').loc[x,'subject'] in s288c_genes]
all_genes = ica_data_cenpk.X.index

def categorize_genes(x):
    if x in overlap:
        return 'Has Ortholog in S288C'
    elif x in cenpk_genes and x in bbh.gene.values:
        return 'Ortholog not in S288C Component'
    elif x in cenpk_genes:
        return "No Known Ortholog"
    return 'Not in iM'

groups = {x:categorize_genes(x) for x in all_genes}
kwargs = {'groups':groups, 'colors':{'Has Ortholog in S288C': 'orange', "No Known Ortholog":'tab:blue', 'Not in iM':'Gray', 'Ortholog not in S288C Component':'tab:purple'}}

plot_gene_weights_chrom(ica_data_cenpk, im, **kwargs)
plt.xticks(rotation = 45);


plt.title('Cen.PK GCN4 iModulon');
plt.savefig('images/fig5_GCN4_cenpk.svg', dpi = 600, format = 'svg', bbox_inches='tight')

In [ ]:
from matplotlib_venn import venn2
# Define the sizes of the regions
size_only_A = len(cenpk_genes) - len(overlap)  # Elements only in Set A
size_only_B = len(s288c_genes) - len(overlap)   # Elements only in Set B
size_A_and_B = len(overlap)  # Elements in both Set A and Set B

# Create the Venn diagram
venn2(subsets=(size_only_A, size_only_B, size_A_and_B), set_labels=(f'CEN.PK {im} iM Genes', f'S288C {im} iM Genes'))

plt.title('CEN.PK GCN4 Overlap With S288C');


plt.savefig('images/fig5_GCN4_cenpk_vs_s288c_venn.svg', dpi = 600, format = 'svg', bbox_inches='tight')

# Display the diagram
plt.show()

# Display the diagram
plt.show()

# Activity PCA of Dataset

In [ ]:
# Extract data and metadata
A = ica_data_cenpk.A
metadata = ica_data_cenpk.sample_table

strain_map = {
    'IMX2816':'CEN.PK113-7D',
    'CEN.PK 2-1C':'CEN.PK 2-1C',
    'CEN.PK 113-7D':'CEN.PK 113-7D',
    'CEN.PK 113-5D':'CEN.PK 113-5D',
    'CEN.PK 113-5D pSP-GM1':'CEN.PK 113-5D',
    'CEN.PK 530.1C':'CEN.PK 530.1C',
    'CEN.PK113-7D': 'CEN.PK 113-7D',
    'CEN.PK-113.5D (KiURA3)':'CEN.PK-113.5D',
    'CEN.PK 102-5B Reference':'CEN.PK 102-5B',
    'CEN.PK 102-5B ST938':'CEN.PK 102-5B',
    'CEN.PK 102-5B':'CEN.PK 102-5B',
    'CEN.PK - St4288':'CEN.PK - St4288',
    'CEN.PK - St4408':'CEN.PK - St4408',
    'CEN.PK':'CEN.PK Nonspecific'
}

metadata['strain'] = metadata.strain_description.apply(lambda x: strain_map[x])

# Perform PCA
pca = PCA(n_components=3)
principal_components = pca.fit_transform(A.T)

# Create PCA dataframe
df_pca = pd.DataFrame(data=principal_components, columns=['PC1', 'PC2', 'PC3'], index=A.columns)

# Get unique projects and assign colors
unique_projects = metadata['strain'].unique()
color_palette = sns.color_palette("tab20", len(unique_projects)).as_hex()
project_color_map = dict(zip(unique_projects, color_palette))

# Set up the plot
fig, ax = plt.subplots(figsize=(8.5, 5))

# Plot each sample
for index, row in metadata.iterrows():
    project = row['strain']
    color = project_color_map.get(project, 'gray')

    ax.scatter(
        df_pca.loc[index, 'PC1'],
        df_pca.loc[index, 'PC2'],
        color=color,
        s=200,
        alpha=0.7,
        edgecolors='k',
        linewidths=.1
    )

# Axis labels with explained variance
ax.set_xlabel(f'PC1 - Explained Variance: {pca.explained_variance_ratio_[0]:.2%}', fontsize=20)
ax.set_ylabel(f'PC2 - Explained Variance: {pca.explained_variance_ratio_[1]:.2%}', fontsize=20)

# Biplot arrows
loadings = pd.DataFrame(index=A.index)
loadings['PC1'] = pca.components_[0, :]
loadings['PC2'] = pca.components_[1, :]

arrows_to_make = list(loadings['PC1'].abs().sort_values(ascending=False).index[:7]) + \
                 list(loadings['PC2'].abs().sort_values(ascending=False).index[:1])
arrows_to_make = sorted(set(arrows_to_make))

arrow_length = 50
text_scale = 1.1
start1, start2 = 0, 0

for g in arrows_to_make:
    g_weight_pc1 = loadings.loc[g, 'PC1'] * arrow_length
    g_weight_pc2 = loadings.loc[g, 'PC2'] * arrow_length

    ax.arrow(start1, start2, g_weight_pc1, g_weight_pc2, head_width=1, color='black')
    ax.text(start1 + (text_scale * g_weight_pc1) + 1,
            start2 + (text_scale * g_weight_pc2),
            g, size=15)

# Create legend by project
legend_elements = [
    plt.Line2D([0], [0],
               marker='o',
               color=color,
               label=project,
               markersize=10,
               linestyle='None',
               markerfacecolor=color,
               alpha=0.7,
               markeredgecolor='k',
               linewidth=.1
              )
    for project, color in project_color_map.items()
]

plt.legend(legend_elements, project_color_map.keys(), title='Labeled Strain', loc='upper left', fontsize=8, ncol=2)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.title("Activity PCA Plot Colored by Strain", fontsize=24)
plt.tight_layout()

plt.savefig('images/fig5_cenpk_activity_pca_by_project.svg', format='svg')

# Statistics for Number of orthologs in CEN.PK iModulons

In [ ]:
overlap_with_s288c = pd.Series()
for im in ica_data_cenpk.imodulon_names:
    genes_with_ortholog = ica_data_cenpk.view_imodulon(im).S288C_ID.dropna().astype(bool).sum()
    total = ica_data_cenpk.view_imodulon(im).shape[0]
    overlap_with_s288c[im] = genes_with_ortholog/total

In [ ]:
overlap_with_s288c.sort_values()

In [ ]:
im = 'HAP4'
s288c_genes = ica_data.view_imodulon(im).index
cenpk_genes = ica_data_cenpk.view_imodulon(im).index
overlap = [x for x in ica_data_cenpk.view_imodulon(im).index if x in bbh.gene.values and bbh.set_index('gene').loc[x,'subject'] in s288c_genes]
all_genes = ica_data_cenpk.X.index

def categorize_genes(x):
    if x in overlap:
        return 'Has Ortholog in S288C'
    elif x in cenpk_genes and x in bbh.gene.values:
        return 'Ortholog not in S288C Component'
    elif x in cenpk_genes:
        return "No Known Ortholog"
    return 'Not in iM'

groups = {x:categorize_genes(x) for x in all_genes}
kwargs = {'groups':groups, 'colors':{'Has Ortholog in S288C': 'orange', "No Known Ortholog":'tab:blue', 'Not in iM':'Gray', 'Ortholog not in S288C Component':'tab:purple'}}

plot_gene_weights_chrom(ica_data_cenpk, im, **kwargs)
plt.xticks(rotation = 45);


plt.title('Cen.PK Sulfur Amino Acid Metabolism iModulon');

# Activities of All Overlaping iMs with S288C and CEN.PK

In [ ]:
import numpy as np

for link1, link2, score in links:
    fig, ax = plt.subplots(1, 1)

    data1 = ica_data.A.loc[link2]
    data2 = ica_data_cenpk.A.loc[link1]

    # Combine data to compute shared bin edges
    combined = np.concatenate([data1.values, data2.values])

    bins = np.histogram_bin_edges(combined, bins=30)

    ax.hist(data1,
            bins=bins,
            alpha=1.0,
            label=f"S288C - {link2}",
            density=True)

    ax.hist(data2,
            bins=bins,
            alpha=0.5,
            label=f"CEN.PK - {link1}",
            density=True)

    ax.legend()
    ax.set_xlabel("Value")
    ax.set_ylabel("Density")
    ax.set_title(f"Histogram comparison (score={score})")

    plt.show()


In [ ]:
ica_data.A.T.corr()['IFH1'].sort_values()

In [ ]:
ica_data.A.T.corr()['IFH1'].sort_values()['GCN4 Associated']

In [ ]:
ica_data_cenpk.A.T.corr()['IFH1'].sort_values()['GCN4 Associated']

In [ ]:
im = 'GCN4 Associated'
s288c_genes = ica_data.view_imodulon(im).index
cenpk_genes = ica_data_cenpk.view_imodulon(im).index
overlap = [x for x in ica_data_cenpk.view_imodulon(im).index if x in bbh.gene.values and bbh.set_index('gene').loc[x,'subject'] in s288c_genes]
all_genes = ica_data_cenpk.X.index

def categorize_genes(x):
    if x in overlap:
        return 'Has Ortholog in S288C'
    elif x in cenpk_genes and x in bbh.gene.values:
        return 'Ortholog not in S288C Component'
    elif x in cenpk_genes:
        return "No Known Ortholog"
    return 'Not in iM'

groups = {x:categorize_genes(x) for x in all_genes}
kwargs = {'groups':groups, 'colors':{'Has Ortholog in S288C': 'orange', "No Known Ortholog":'tab:blue', 'Not in iM':'Gray', 'Ortholog not in S288C Component':'tab:purple'}}

plot_gene_weights_chrom(ica_data_cenpk, im, **kwargs)
plt.xticks(rotation = 45);


plt.title(f'Cen.PK {im} iModulon');

In [ ]:
plot_gene_weights_chrom(ica_data, 'GCN4 Associated')

In [ ]:
cenpk_s288c_im_corrs = []

cenpk_genes = ica_data_cenpk.gene_names
s288c_genes = ica_data.gene_names
shared = bbh[bbh.gene.isin(cenpk_genes) & bbh.subject.isin(s288c_genes)]

for cenpk_im, s288c_im, score in links:
    cenpk_s288c_im_corrs.append([cenpk_im, s288c_im, pearsonr(ica_data_cenpk.M.loc[shared.gene.values, cenpk_im], ica_data.M.loc[shared.subject.values, s288c_im])[0]])


cenpk_s288c_im_corrs = pd.DataFrame(cenpk_s288c_im_corrs, columns = ['cenpk_im', 's288c_im', 'corr'])
cenpk_s288c_im_corrs.sort_values('corr', key = abs)

In [ ]:
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests

# --- 1. Setup Sample Sizes ---
# Standard error for Fisher's Z depends on the number of samples (n)
n_s288c = ica_data.A.shape[1]
n_cenpk = ica_data_cenpk.A.shape[1]

# --- 2. Calculate Correlations and Align ---
corrs_s288c = ica_data.A.T.corr().loc['IFH1']
corrs_cenpk = ica_data_cenpk.A.T.corr().loc['IFH1']

link_df = cenpk_s288c_im_corrs.sort_values('corr', key=abs, ascending=False).drop_duplicates('cenpk_im')
link_df = link_df.sort_values('corr', key=abs, ascending=False).drop_duplicates('s288c_im')

plot_data = []
for _, row in link_df.iterrows():
    s28, cen = row['s288c_im'], row['cenpk_im']
    
    if s28 in corrs_s288c.index and cen in corrs_cenpk.index:
        r1 = corrs_s288c[s28]
        r2_raw = corrs_cenpk[cen]
        
        # Align r2 based on the similarity sign
        sign = row['corr'] / abs(row['corr'])
        r2 = sign * r2_raw

        # 1. Convert r to z (clip to avoid inf if r=1)
        z1 = np.arctanh(np.clip(r1, -0.999, 0.999))
        z2 = np.arctanh(np.clip(r2, -0.999, 0.999))
        
        # 2. Standard Error of the difference
        se = np.sqrt(1/(n_s288c - 3) + 1/(n_cenpk - 3))
        
        # 3. Z-statistic and P-value
        z_diff = (z1 - z2) / se
        p_val = 2 * (1 - norm.cdf(abs(z_diff))) # Two-tailed
        
        plot_data.append({
            'S288C': s28, 'CENPK': cen,
            'x': r1, 'y': r2,
            'Similarity': row['corr'],
            'p_value': p_val,
            'z_diff': z_diff,
            'Dist': abs(r1 - r2)
        })

df = pd.DataFrame(plot_data)

# --- 3. Multiple Testing Correction ---
# Using Benjamini-Hochberg (FDR)
_, p_adj, _, _ = multipletests(df['p_value'], method='fdr_bh')
df['p_adj'] = p_adj
df['is_significant'] = df['p_adj'] < 0.05

# --- 4. Plotting ---
plt.figure(figsize=(8, 5))

# Shading and Reference lines
plt.axvspan(0, 1.2, 0, 1.2, color='green', alpha=0.03)
plt.axvspan(-1.2, 0, -1.2, 0, color='red', alpha=0.03)
plt.plot([-1.2, 1.2], [-1.2, 1.2], color='black', linestyle='--', alpha=0.2)
plt.axhline(0, color='black', lw=1, alpha=0.3)
plt.axvline(0, color='black', lw=1, alpha=0.3)

# Scatter plot: 
# Points are colored by similarity of iMs
df['neg_log_p'] = -np.log10(df['p_adj'] + 1e-10)
scatter = plt.scatter(df['x'], df['y'], 
                      c=df['Similarity'], cmap='Reds', s = 100,
                      edgecolors='white', alpha=0.8, zorder=3,
                      vmin = .2, vmax = .75
                     )

# Labeling
for i, row in df.iterrows():
    if row['is_significant'] and row['Dist'] > 0.8:
        weight = 'bold' if row['is_significant'] else 'normal'
        color = 'red' if row['is_significant'] else 'black'
        plt.text(row['x']+0.02, row['y'], row['S288C'], 
                 fontsize=9, fontweight=weight, color=color)
    else:
        plt.text(row['x']+0.02, row['y'], row['S288C'], fontsize=9)

plt.title('Fear vs. Greed iModulon Coordination Divergence', fontsize=16, fontweight='bold')
plt.xlabel('Correlation with IFH1 (S288C)', fontsize=13)
plt.ylabel('Correlation with IFH1 (CEN.PK)', fontsize=13)
plt.colorbar(scatter, label='iModulon Similarity')
plt.grid(True, linestyle=':', alpha=0.4)
plt.savefig('images/fig5_orthologous_im_fear_greed_comparison.svg', dpi = 600, format = 'svg', bbox_inches='tight')
plt.show()

In [ ]:
im = 'GCN4 Associated'
# 1. Extract GCN4 Associated gene weights from both models
# S288C weights
s288c_weights = ica_data.M[im].to_frame(name='S288C_Weight')
# CEN.PK weights
cenpk_weights = ica_data_cenpk.M[im].to_frame(name='CENPK_Weight')

# 2. Align weights using the ortholog table (bbh)
# bbh mapping: 'gene' column is CEN.PK, 'subject' column is S288C
df_weights = bbh[['gene', 'subject']].copy()
df_weights = df_weights.merge(s288c_weights, left_on='subject', right_index=True)
df_weights = df_weights.merge(cenpk_weights, left_on='gene', right_index=True)

# Add gene names from the S288C gene table for better labeling
if 'gene_name' in ica_data.gene_table.columns:
    df_weights = df_weights.merge(ica_data.gene_table[['gene_name']], left_on='subject', right_index=True)
else:
    df_weights['gene_name'] = df_weights['subject']

# 3. Identify iModulon members
s288c_members = ica_data.view_imodulon(im).index
cenpk_members = ica_data_cenpk.view_imodulon(im).index

def get_membership(row):
    in_s28 = row['subject'] in s288c_members
    in_cen = row['gene'] in cenpk_members
    if in_s28 and in_cen: return 'Both'
    if in_s28: return 'S288C Only'
    if in_cen: return 'CEN.PK Only'
    return 'Neither'

df_weights['Membership'] = df_weights.apply(get_membership, axis=1)

# 4. Generate the Plot
plt.figure(figsize=(8, 8))
sns.set_style('whitegrid')

# Plot non-members in light gray
sns.scatterplot(data=df_weights[df_weights['Membership'] == 'Neither'], 
                x='S288C_Weight', y='CENPK_Weight', color='lightgray', alpha=0.3, label='Other Orthologs', zorder=1, rasterized=True)

# Plot members with distinct colors
colors = {'Both': '#551a8b', 'S288C Only': '#e41a1c', 'CEN.PK Only': '#377eb8'}
for group, color in colors.items():
    subset = df_weights[df_weights['Membership'] == group]
    plt.scatter(subset['S288C_Weight'], subset['CENPK_Weight'], 
                color=color, label=f'In {group} iModulon', s=60, edgecolors='white', alpha=0.8, zorder=2)

# Add reference line (Diagonal)
max_val = df_weights[['S288C_Weight', 'CENPK_Weight']].abs().max().max()
plt.plot([-max_val, max_val], [-max_val, max_val], 'k--', alpha=0.2, zorder=0)
plt.axhline(0, color='black', lw=0.5); plt.axvline(0, color='black', lw=0.5)

# # 5. Labeling logic
# # Filter for genes to label (e.g., top 15 by weight in either strain)
# label_threshold = 0.05
# to_label = df_weights[(df_weights['S288C_Weight'].abs() > label_threshold) | 
#                       (df_weights['CENPK_Weight'].abs() > label_threshold)].copy()

# for i, row in to_label.sort_values('S288C_Weight', ascending=False).head(15).iterrows():
#     plt.text(x=row['S288C_Weight'] + 0.002, 
#              y=row['CENPK_Weight'], 
#              s=row['gene_name'], 
#              fontsize=9, 
#              fontweight='bold' if row['Membership'] == 'Both' else 'normal')

# Calculate correlation
corr = df_weights['S288C_Weight'].corr(df_weights['CENPK_Weight'])
plt.title(f'{im} Gene Weights: S288C vs CEN.PK\nPearson R = {corr:.3f}', fontsize=14, fontweight='bold')
plt.xlabel('S288C Gene Weight (S288C Model)', fontsize=12)
plt.ylabel('CEN.PK Gene Weight (CEN.PK Model)', fontsize=12)
plt.legend(title='iModulon Membership', loc='lower right')

plt.tight_layout()
plt.savefig('images/fig5_GCN4_orthologous_gene_weights.svg', dpi = 600, format = 'svg', bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

im = 'IFH1'
# S288C weights
s288c_weights = ica_data.M[im].to_frame(name='S288C_Weight')
# CEN.PK weights
cenpk_weights = ica_data_cenpk.M[im].to_frame(name='CENPK_Weight')

# 2. Align weights using the ortholog table (bbh)
# bbh mapping: 'gene' column is CEN.PK, 'subject' column is S288C
df_weights = bbh[['gene', 'subject']].copy()
df_weights = df_weights.merge(s288c_weights, left_on='subject', right_index=True)
df_weights = df_weights.merge(cenpk_weights, left_on='gene', right_index=True)

# Add gene names from the S288C gene table for better labeling
if 'gene_name' in ica_data.gene_table.columns:
    df_weights = df_weights.merge(ica_data.gene_table[['gene_name']], left_on='subject', right_index=True)
else:
    df_weights['gene_name'] = df_weights['subject']

# 3. Identify iModulon members
s288c_members = ica_data.view_imodulon(im).index
cenpk_members = ica_data_cenpk.view_imodulon(im).index

def get_membership(row):
    in_s28 = row['subject'] in s288c_members
    in_cen = row['gene'] in cenpk_members
    if in_s28 and in_cen: return 'Both'
    if in_s28: return 'S288C Only'
    if in_cen: return 'CEN.PK Only'
    return 'Neither'

df_weights['Membership'] = df_weights.apply(get_membership, axis=1)

# 4. Generate the Plot
plt.figure(figsize=(8, 8))
sns.set_style('whitegrid')

# Plot non-members in light gray
sns.scatterplot(data=df_weights[df_weights['Membership'] == 'Neither'], 
                x='S288C_Weight', y='CENPK_Weight', color='lightgray', alpha=0.3, label='Other Orthologs', zorder=1, rasterized = True)

# Plot members with distinct colors
colors = {'Both': '#551a8b', 'S288C Only': '#e41a1c', 'CEN.PK Only': '#377eb8'}
for group, color in colors.items():
    subset = df_weights[df_weights['Membership'] == group]
    plt.scatter(subset['S288C_Weight'], subset['CENPK_Weight'], 
                color=color, label=f'In {group} iModulon', s=60, edgecolors='white', alpha=0.8, zorder=2)

# Add reference line (Diagonal)
max_val = df_weights[['S288C_Weight', 'CENPK_Weight']].abs().max().max()
plt.plot([-max_val, max_val], [-max_val, max_val], 'k--', alpha=0.2, zorder=0)
plt.axhline(0, color='black', lw=0.5); plt.axvline(0, color='black', lw=0.5)

# # 5. Labeling logic
# # Filter for genes to label (e.g., top 15 by weight in either strain)
# label_threshold = 0.05
# to_label = df_weights[(df_weights['S288C_Weight'].abs() > label_threshold) | 
#                       (df_weights['CENPK_Weight'].abs() > label_threshold)].copy()

# for i, row in to_label.sort_values('S288C_Weight', ascending=False).head(15).iterrows():
#     plt.text(x=row['S288C_Weight'] + 0.002, 
#              y=row['CENPK_Weight'], 
#              s=row['gene_name'], 
#              fontsize=9, 
#              fontweight='bold' if row['Membership'] == 'Both' else 'normal')

# Calculate correlation
corr = df_weights['S288C_Weight'].corr(df_weights['CENPK_Weight'])
plt.title(f'{im} Gene Weights: S288C vs CEN.PK\nPearson R = {corr:.3f}', fontsize=14, fontweight='bold')
plt.xlabel('S288C Gene Weight (S288C Model)', fontsize=12)
plt.ylabel('CEN.PK Gene Weight (CEN.PK Model)', fontsize=12)
plt.legend(title='iModulon Membership', loc='lower right')

plt.tight_layout()
plt.savefig('images/fig5_ifh1_orthologous_gene_weights.svg', dpi = 600, format = 'svg', bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

# 1. Define the iModulons to compare
growth_im = 'IFH1'
met_im = 'GCN4 Associated'

# --- FIGURE 1: S288C Phase Plane ---
plt.figure(figsize=(6, 6))
x_s = ica_data.A.loc[growth_im]
y_s = ica_data.A.loc[met_im]
r_s, p_s = pearsonr(x_s, y_s)

sns.regplot(x=x_s, y=y_s, 
            scatter_kws={'alpha':0.4, 's':40, 'color':'gray'}, 
            line_kws={'color':'green', 'lw':2, 'label':f'Pearson r = {r_s:.2f}\np = {p_s:.1e}'})

plt.title(f'S288C: {met_im} iM vs {growth_im} iM', fontsize=14, fontweight='bold')
plt.xlabel(f'{growth_im} iM Activity (Growth)', fontsize=12)
plt.ylabel(f'{met_im} iM Activity', fontsize=12)
plt.legend(frameon=True)
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.savefig('images/fig5_S288C_GCN4_PhasePlane.svg', dpi=300) # Optional: save for paper
plt.show()


# --- FIGURE 2: CEN.PK Phase Plane ---
plt.figure(figsize=(6, 6))
x_c = ica_data_cenpk.A.loc[growth_im]
y_c = ica_data_cenpk.A.loc[met_im]
r_c, p_c = pearsonr(x_c, y_c)

sns.regplot(x=x_c, y=y_c, 
            scatter_kws={'alpha':0.4, 's':40, 'color':'gray'}, 
            line_kws={'color':'red', 'lw':2, 'label':f'Pearson r = {r_c:.2f}\np = {p_c:.1e}'})

plt.title(f'CEN.PK: {met_im} iM vs {growth_im} iM', fontsize=14, fontweight='bold')
plt.xlabel(f'{growth_im} iM Activity (Growth)', fontsize=12)
plt.ylabel(f'{met_im} iM Activity', fontsize=12)
plt.legend(frameon=True)
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.savefig('images/fig5_CENPK_GCN4_PhasePlane.svg', dpi=300) # Optional: save for paper
plt.show()

# Clustermap diagrams of each strain

In [ ]:
link_df = (
    cenpk_s288c_im_corrs
    .sort_values('corr', key=abs, ascending=False)
    .drop_duplicates('cenpk_im')
)

link_df = (
    link_df
    .sort_values('corr', key=abs, ascending=False)
    .drop_duplicates('s288c_im')
)

In [ ]:
ica_a_corr = ica_data.A.T.corr()

fear_greed_ims = (
    ica_a_corr.loc['IFH1']
    [ica_a_corr.loc['IFH1'].abs() > .7]
    .index
)

shared = []


for _, row in link_df.iterrows():
    if (
        abs(corrs_s288c[row.s288c_im]) > 0.7 or
        abs(corrs_cenpk[row.cenpk_im]) > 0.7
    ):
        shared.append(row)

s288c_order = [r.s288c_im for r in shared]
cenpk_order = [r.cenpk_im for r in shared]

corr1 = (
    ica_data.A
    .loc[s288c_order]
    .T.corr()
)

corr2 = (
    ica_data_cenpk.A
    .loc[cenpk_order]
    .T.corr()
)


g = sns.clustermap(corr1)
plt.savefig('images/fig5_s288c_fear_greed_clustermap.svg', dpi = 600, format = 'svg', bbox_inches='tight')


In [ ]:
g = sns.clustermap(corr2)
plt.savefig('images/fig5_cenpk_fear_greed_clustermap.svg', dpi = 600, format = 'svg', bbox_inches='tight')